In [1]:
# | export

import json
import logging
import warnings
import webbrowser
from pathlib import Path

import hvplot
import pooch
import rasterio
import rioxarray as rxr
from dask import compute, delayed
from fastcore.script import call_parse
from yarl import URL

hvplot.extension(inline=False)
import hvplot.xarray  # noqa
from fastcore.utils import Path
from planetarypy.config import config
from planetarypy.pds import get_index
from planetarypy.instruments import hirise
from planetarypy.utils import check_url_exists, url_retrieve

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

In [2]:
edrindex = get_index("mro.hirise.edr")

In [4]:
edrindex.PRODUCT_ID.head()

0    AEB_000001_0000_BG13_0
1    AEB_000001_0000_BG13_1
2    AEB_000001_0000_RED1_0
3    AEB_000001_0000_RED1_1
4    AEB_000001_0000_RED3_0
Name: PRODUCT_ID, dtype: string

In [5]:
# | export
class OBSID:
    """Manage HiRISE observation ids.

    For example PSP_003092_0985.
    `phase` is set to PSP for orbits < 11000, no setting required.
    """

    def __init__(
        self,
        obsid: str,  # e.g. PSP_003092_0985
    ):
        phase, orbit, targetcode = obsid.split("_")
        self._orbit = int(orbit)
        self._targetcode = targetcode

    @property
    def orbit(self):
        return str(self._orbit).zfill(6)

    @orbit.setter
    def orbit(self, value: int):  # e.g. 11000, < 1_000_000
        if value > 999999:
            raise ValueError("Orbit cannot be larger than 999999")
        elif len(value) != 6:
            raise ValueError("Orbit string must be 6 digits.")
        self._orbit = value

    @property
    def targetcode(self):
        return self._targetcode

    @targetcode.setter
    def targetcode(
        self,
        value: str,  # e.g. "0985", must be 4 digits
    ):
        if len(str(value)) != 4:
            raise ValueError("Targetcode must be exactly 4 characters.")
        self._targetcode = value

    @property
    def phase(self):
        return "PSP" if int(self.orbit) < 11000 else "ESP"

    @property
    def id(self):
        return f"{self.phase}_{self.orbit}_{self.targetcode}"

    def __repr__(self):
        return self.__str__()

    def __str__(self):
        return self.id

    @property
    def upper_orbit_folder(self):
        """
        get the upper folder name wa
        rdrhere the given orbit folder is residing on the
        hisync server, e.g. 'ORB_011900_011999'
        """
        lower = int(self.orbit) // 100 * 100
        lowerstr = str(lower).zfill(6)
        upperstr = str(lower + 99).zfill(6)
        return f"ORB_{lowerstr}_{upperstr}"

    @property
    def storage_path_stem(self):
        return f"{self.phase}/{self.upper_orbit_folder}/{self.id}"

    @property
    def rdr_path(self):
        return Path("RDR") / self.storage_path_stem

    @property
    def rdr_url(self):
        return baseurl / str(self.rdr_path)
